# Movie Recommendation System
This notebook demonstrates the implementation of a content-based movie recommendation system using the TMDB dataset, including movie poster integration.

## 1. Importing Libraries and Loading Data

In [ ]:
import pandas as pd
import ast
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

In [ ]:
# Load datasets
movies = pd.read_csv('Data/movies.csv')
credits = pd.read_csv('Data/credits.csv')
posters = pd.read_csv('Data/poster.csv')

print("Datasets loaded successfully.")

## 2. Data Cleaning and Merging
We will merge the `movies`, `credits`, and `poster` datasets on the movie title.

In [ ]:
# Merge datasets on title
movies = movies.merge(credits, on='title')
movies = movies.merge(posters, on='title')

# Select relevant columns
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew', 'poster']]

# Check for missing values
movies.dropna(inplace=True)
movies.head()

## 3. Feature Engineering
Extract relevant information from JSON-like structures in `genres`, `keywords`, `cast`, and `crew`.

In [ ]:
def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

In [ ]:
def convert3(obj):
    L = []
    counter = 0
    for i in ast.literal_eval(obj):
        if counter != 3:
            L.append(i['name'])
            counter += 1
        else:
            break
    return L

movies['cast'] = movies['cast'].apply(convert3)

In [ ]:
def fetch_director(obj):
    L = []
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            L.append(i['name'])
            break
    return L

movies['crew'] = movies['crew'].apply(fetch_director)

In [ ]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())

### Removing spaces from tags

In [ ]:
movies['genres'] = movies['genres'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['cast'] = movies['cast'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['crew'] = movies['crew'].apply(lambda x: [i.replace(" ", "") for i in x])

## 4. Creating the Similarity Model

In [ ]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']
new_df = movies[['movie_id', 'title', 'tags', 'poster']]
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())
new_df.head()

In [ ]:
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()
similarity = cosine_similarity(vectors)

## 5. Recommendation System in Action

In [ ]:
def recommend(movie):
    movie_index = new_df[new_df['title'] == movie].index[0]
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]
    
    print(f"Recommendations for '{movie}':")
    for i in movies_list:
        m = new_df.iloc[i[0]]
        print(f"- {m.title} (Poster: {m.poster})")

recommend('Avatar')

## 6. Saving the Model

In [ ]:
pickle.dump(new_df, open('movie_list.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))